In [88]:
import scipy.io
import pandas as pd
import h5py
import numpy as np
import pickle
from pathlib import Path
import glob

In [89]:
# Update the session variable to match the date in your file names
session = '10/09/2025'

YY = session[-2:]
YYYY = session[-4:]
MM = session[3:5]
DD = session[:2]

DD_MM_YYYY = f"{DD}_{MM}_{YYYY}"
MM_DD_YYYY = f"{MM}_{DD}_{YYYY}"

In [100]:
spike_clusters = scipy.io.loadmat(rf'/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/aligned_spike_clusters.mat')
spike_times = scipy.io.loadmat(rf'/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/aligned_spike_times.mat')
cluster_info = pd.read_csv(rf'/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/cluster_info.tsv', sep='\t')
file_data = pd.read_csv(rf"/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/NPC1_10092025_1_g0_ct_offsets.txt", sep='\t')


In [91]:
improved_node_summary_csv = f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/node_summary_df.csv"
summary_df_pkl = f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/summary_df.pkl"

# Load the DataFrames from the csv and pickle files
with open(improved_node_summary_csv, 'rb') as f:
    improved_node_summary_df = pd.read_csv(f)

with open(summary_df_pkl, 'rb') as f:
    summary_df = pickle.load(f)

In [92]:
import networkx as nx
# Load the mapping file to get node positions
mapping_file = Path('/var/home/almogmeir/Documents/GitHub/NaviGraph/FullMazeGrid.pkl')
with open(mapping_file, 'rb') as f:
    mapping_data = pickle.load(f)

# Extract node and edge definitions
nodes_dict = mapping_data['mappings']['nodes']
edges_dict = mapping_data['mappings']['edges']

print(f"Found {len(nodes_dict)} nodes in mapping")
print(f"Found {len(edges_dict)} edge definitions")

# Convert node polygons to center positions
node_positions = {}
for node_id, polygon_list in nodes_dict.items():
    polygon = polygon_list[0] if polygon_list else []
    
    if polygon and len(polygon) >= 2:
        # Calculate center of bounding box
        xs = [p[0] for p in polygon]
        ys = [p[1] for p in polygon]
        center_x = sum(xs) / len(xs)
        center_y = sum(ys) / len(ys)
        node_positions[node_id] = (center_x, center_y)

print(f"Converted {len(node_positions)} node positions")

# Build graph structure from edges_dict
G = nx.DiGraph()

# Add all nodes
G.add_nodes_from(node_positions.keys())

# Add edges from edges_dict (bidirectional)
edges_added = 0
for edge_key, polygon_list in edges_dict.items():
    nodes = edge_key.split('_')
    if len(nodes) == 2:
        from_node, to_node = nodes
        if from_node in node_positions and to_node in node_positions:
            G.add_edge(from_node, to_node)
            G.add_edge(to_node, from_node)
            edges_added += 2

print(f"\nGraph loaded: {len(G.nodes)} nodes, {edges_added} edges (bidirectional)")
print(f"Graph ready for visualizations!")

Found 126 nodes in mapping
Found 125 edge definitions
Converted 126 node positions

Graph loaded: 126 nodes, 250 edges (bidirectional)
Graph ready for visualizations!


In [24]:
# The mapping based on your pairs

if MM == '09' and DD in {'07', '08', '09', '10', '11'}:
    print("Session After 07/09, using port_map after switch")
    port_map = {
        'R522': 'Target1', 'R526': 'Target2',
        'L56':  'Target3', 'L510': 'Target4',
        'L521': 'Target5', 'L525': 'Target6',
        'R59':  'Target7', 'R55':  'Target8'
    }
    pairs_to_check = [('R522', 'R526'), ('L56', 'L510'), ('L521', 'L525'), ('R59', 'R55')]


else: 
    print("Session Before 07/09, using port_map before switch")
    port_map = {
        'R522': 'Target1', 'R526': 'Target2',
        'L56':  'Target3', 'L510': 'Target4',
        'L521': 'Target5', 'L525': 'Target6',
        'R55':  'Target7', 'R59':  'Target8'
    }
    pairs_to_check = [('R522', 'R526'), ('L56', 'L510'), ('L521', 'L525'), ('R55', 'R59')]

Session After 07/09, using port_map after switch


In [94]:
improved_node_summary_df

,node_name,node_name_numeric,start_frame,end_frame,duration_frames,reward_during_node,reward_size_during_node,lick_during_node,lick_count_during_node,is_port_node,trial_idx,node_visit_idx,path_start_time,port_to_port_path_type,path_pair_label,path_seq_length_unique,path_shortest_length
0,L31,8,12,12,1,False,0.0,False,0,False,0,0,NaN,NaN,NaN,NaN,NaN
1,L20,3,14,14,1,False,0.0,False,0,False,0,1,NaN,NaN,NaN,NaN,NaN
2,L30,7,16,16,1,False,0.0,False,0,False,0,2,NaN,NaN,NaN,NaN,NaN
3,L40,15,18,18,1,False,0.0,False,0,False,0,3,NaN,NaN,NaN,NaN,NaN
4,L50,29,19,20,2,False,0.0,False,0,False,0,4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5722,L35,12,145303,145305,3,False,0.0,False,0,False,736,5722,3619.725,direct,"('Target4', 'Target5')",10.0,10.0
5723,L410,17,145317,145324,8,False,0.0,False,0,False,736,5723,3619.725,direct,"('Target4', 'Target5')",10.0,10.0
5724,L521,39,145325,145345,21,False,0.0,False,0,True,736,5724,3619.725,direct,"('Target4', 'Target5')",10.0,10.0
5725,L410,17,145346,145401,9,False,0.0,False,0,False,736,5725,NaN,NaN,NaN,NaN,NaN


In [95]:
summary_df

headstage                              ear_L                         \
                 x           y likelihood           x           y likelihood   
0       433.929993  132.995728   0.740949  450.688568  125.864143   0.785938   
1       440.402649  128.830093   0.783941  451.066956  123.661530   0.827651   
2       440.582001  141.402664   0.632677  451.312408  126.520264   0.765058   
3       432.078552  150.250290   0.580932  448.136017  134.566772   0.618934   
4       432.772217  153.065552   0.658617  449.797272  134.214859   0.673868   
...            ...         ...        ...         ...         ...        ...   
144681  155.169388  760.776917   0.944158  158.959015  776.431946   0.600485   
144682  155.364243  762.666443   0.963189  157.732925  776.007202   0.665001   
144683  156.229050  761.671936   0.982746  158.886581  777.108459   0.660428   
144684  156.117126  761.662903   0.943683  159.101593  778.244812   0.653442   
144685  156.361084  759.098877   0.918868  162.950851  784.482056   0.574168   

             ear_R                            midbody  ...  \
                 x           y likelihood           x  ...   
0       440.674438  109.750801   0.644472  466.256317  ...   
1       441.147614  111.808052   0.776953  462.443909  ...   
2       438.411926  116.412804   0.745470  462.961426  ...   
3       438.771240  118.233482   0.589876  462.641754  ...   
4       434.065582  125.292442   0.595024  459.893372  ...   
...            ...         ...        ...         ...  ...   
144681  153.356842  771.957947   0.734815  160.023331  ...   
144682  154.920090  772.296082   0.769864  159.963516  ...   
144683  156.001587  772.521545   0.748313  160.321030  ...   
144684  155.535461  773.272827   0.705107  160.255951  ...   
144685  155.310516  773.524597   0.640854  162.040390  ...   

       frame_time_in_trial trial_length is_exact_multiple has_extra_frame  \
                                                                            
0                    0.000       9.7590             False            True   
1                    0.025       9.7590             False            True   
2                    0.050       9.7590             False            True   
3                    0.075       9.7590             False            True   
4                    0.100       9.7590             False            True   
...                    ...          ...               ...             ...   
144681              42.725      42.8674             False            True   
144682              42.750      42.8674             False            True   
144683              42.775      42.8674             False            True   
144684              42.800      42.8674             False            True   
144685              42.825      42.8674             False            True   

       headstage_graph_node headstage_graph_edge fixed_node reward_node  \
                                                                          
0                       NaN           (L31, L43)        NaN       False   
1                       NaN           (L31, L43)        NaN       False   
2                       NaN           (L31, L43)        NaN       False   
3                       NaN           (L31, L43)        NaN       False   
4                       NaN                 None        NaN       False   
...                     ...                  ...        ...         ...   
144681                 L520                 None        L54       False   
144682                 L520                 None        L54       False   
144683                 L520                 None        L54       False   
144684                 L520                 None        L54       False   
144685                 L520                 None        L54       False   

       reward_size   lick  
                           
0              NaN  False  
1              NaN  False  
2              NaN  False  
3              NaN  False  
4              

## Step 1: Extract long indirect paths and find direct-path opportunities
Filter for indirect paths that take 4+ extra nodes beyond the shortest path, then identify where the mouse could have started taking a direct (shortest) path to the destination.

## Step 1 (Corrected): Extract unique long indirect paths
Identify one row per unique path traversal (not once per node), then find where each mouse could have started taking direct paths.

In [96]:
# FPS used to convert frames -> seconds (matches align_frames_trials.ipynb)
FPS = 40

# Step 1: Identify each unique path segment (one row per contiguous run of a
# path_pair_label), keeping only "indirect" paths that are 4+ nodes longer than
# the shortest possible route between the two ports.
df_sorted = improved_node_summary_df.sort_values('node_visit_idx').reset_index(drop=True)

path_segments = []
prev_label = None
for _, row in df_sorted.iterrows():
    label = row['path_pair_label']
    if pd.isna(label):
        prev_label = None
        continue
    if label != prev_label:
        path_segments.append(row)
    prev_label = label

print(f"Total port-to-port path segments in session: {len(path_segments)}")

unique_path_starts = []
for row in path_segments:
    if row['port_to_port_path_type'] != 'indirect':
        continue
    seq_len = int(row['path_seq_length_unique'])
    shortest_len = row['path_shortest_length']
    if (seq_len - shortest_len) < 4:
        continue
    start_idx = int(row['node_visit_idx'])
    # The destination-port row is recorded as the *start* of the next path segment,
    # i.e. exactly one row past the nominal end (start_idx + seq_len - 1). Verified
    # against every one of the 446 path segments in this session: holds with zero
    # exceptions, so we use it directly instead of scanning forward to find it.
    end_idx = start_idx + seq_len
    unique_path_starts.append((start_idx, end_idx, row['path_pair_label'], seq_len, shortest_len, row['trial_idx']))

print(f"Long indirect paths (>=4 excess nodes): {len(unique_path_starts)}\n")

# Step 2: For each path, find the *apex* of the mouse's last detour -- the node from
# which the rest of the trip to the destination is, itself, the direct/shortest route.
#
# The marked node is generally NOT part of the destination's shortest path: it's the
# farthest point of the last detour (e.g. a dead-end side-branch like L520/R531/R50).
# That's expected and correct -- the maze graph is a tree (verified: 126 nodes, 125
# edges, 0 cycles), so there's exactly one way back from any detour. The condition we
# actually want is: "from this node onward, is the remaining trajectory the genuine
# shortest path to the destination?" -- i.e. the apex-to-junction leg is forced/direct
# (always true in a tree) AND the junction-to-destination leg has no further detours.
# We check this with a fresh nx.shortest_path(G, node, dest) per candidate node, and
# take the EARLIEST node in the sequence that satisfies it. Earlier candidates fail
# this check as long as a future detour remains ahead of them, so the first one to
# pass is necessarily the apex of the LAST detour (or, if there was no detour at all
# near the end, the junction node itself).
results = []

for path_idx, (start_nv, end_nv, ppl, seq_len, shortest_len, trial_idx) in enumerate(unique_path_starts):
    seq_rows = improved_node_summary_df[
        (improved_node_summary_df['node_visit_idx'] >= start_nv) &
        (improved_node_summary_df['node_visit_idx'] <= end_nv)
    ].reset_index(drop=True)

    seq_nodes = seq_rows['node_name'].tolist()
    dest_node = seq_nodes[-1]

    direct_found = None
    direct_idx = None
    direct_frame = None
    direct_time_sec = None
    shortest_path_from_node = None
    frames_wasted = None

    for i_node, node in enumerate(seq_nodes):
        try:
            sp = nx.shortest_path(G, node, dest_node)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue
        if seq_nodes[i_node:] == sp:
            direct_found = node
            direct_idx = i_node
            direct_frame = seq_rows.iloc[i_node]['start_frame']
            direct_time_sec = direct_frame / FPS
            shortest_path_from_node = ' → '.join(sp)

            end_frame = seq_rows.iloc[-1]['end_frame']
            frames_available = end_frame - direct_frame
            frames_needed = len(sp) - 1
            frames_wasted = frames_available - frames_needed
            break

    results.append({
        'path_index': path_idx,
        'start_node_visit_idx': start_nv,
        'direct_start_node_visit_idx': start_nv + direct_idx if direct_idx is not None else None,
        'direct_start_position_in_path': direct_idx,
        'path_pair_label': ppl,
        'path_seq_length_unique': seq_len,
        'path_shortest_length': shortest_len,
        'excess_nodes': seq_len - shortest_len,
        'full_sequence': seq_nodes,
        'trial_idx': trial_idx,
        'direct_start_node': direct_found,
        'direct_start_frame': direct_frame,
        'direct_start_time_sec': direct_time_sec,
        'direct_shortest_path_from_node': shortest_path_from_node,
        'frames_wasted': frames_wasted
    })

print(f"Analysis complete: {len(results)} paths analyzed")

# Create dataframe
direct_opportunities_df = pd.DataFrame(results)

def format_path_with_marker(row):
    seq = row['full_sequence']
    marker_pos = row['direct_start_position_in_path']
    if pd.isna(marker_pos):
        return ' → '.join(seq)
    marker_pos = int(marker_pos)
    marked_seq = [f"*{n}*" if i == marker_pos else n for i, n in enumerate(seq)]
    return ' → '.join(marked_seq)

direct_opportunities_df['path_with_marker'] = direct_opportunities_df.apply(format_path_with_marker, axis=1)

print(f"\nFinal unique paths: {len(direct_opportunities_df)}")
print(f"Paths with a committed direct-start node identified: {direct_opportunities_df['direct_start_node'].notna().sum()}")

if not direct_opportunities_df.empty:
    display(direct_opportunities_df[['path_index', 'path_pair_label', 'excess_nodes', 'direct_start_node', 'direct_start_time_sec', 'frames_wasted', 'path_with_marker']].head(25))
else:
    print('No long indirect paths found')

Total port-to-port path segments in session: 521
Long indirect paths (>=4 excess nodes): 17

Analysis complete: 17 paths analyzed

Final unique paths: 17
Paths with a committed direct-start node identified: 17


,path_index,path_pair_label,excess_nodes,direct_start_node,direct_start_time_sec,frames_wasted,path_with_marker
0,0,"('Target5', 'Target6')",14.0,L519,56.850,179,L521 → L410 → L520 → L410 → L35 → L22 → L34 → ...
1,1,"('Target1', 'Target8')",6.0,R519,151.050,142,R522 → R411 → R35 → R22 → R34 → R49 → *R519* →...
2,2,"('Target2', 'Target1')",12.0,R518,654.800,155,R526 → R413 → R36 → R23 → R11 → R22 → R35 → R4...
3,3,"('Target5', 'Target4')",6.0,L412,896.400,459,L521 → L410 → L35 → L22 → L11 → L23 → L36 → *L...
4,4,"('Target4', 'Target8')",6.0,L58,919.175,279,L510 → L45 → L32 → L44 → L58 → L44 → *L58* → L...
5,5,"('Target1', 'Target2')",4.0,R34,1573.400,299,R522 → R411 → R523 → R411 → R35 → R22 → *R34* ...
6,6,"('Target3', 'Target4')",18.0,L51,1626.700,417,L56 → L43 → L31 → L20 → L30 → L41 → L52 → L41 ...
7,7,"('Target1', 'Target2')",6.0,R410,1739.025,203,R522 → R411 → R35 → R22 → R34 → R22 → R35 → *R...
8,8,"('Target6', 'Target5')",10.0,L529,2239.450,413,L525 → L412 → L36 → L23 → L37 → L414 → L529 → ...
9,9,"('Target2', 'Target4')",20.0,L512,2925.075,280,R526 → R413 → R36 → R23 → R11 → R0 → L0 → L11 ...


In [97]:
# Save results to outputs folder for this session
out_dir = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}")
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'direct_opportunities_unique_paths.csv'
direct_opportunities_df.to_csv(out_csv, index=False)
print(f'Saved {len(direct_opportunities_df)} unique paths to {out_csv}')
print(f'  - Paths with direct opportunity: {direct_opportunities_df["direct_start_node"].notna().sum()}')
print(f'  - Paths without direct opportunity: {direct_opportunities_df["direct_start_node"].isna().sum()}')
print(f'  - New column added: path_with_marker (marks direct_start_node with *asterisks*)')

Saved 17 unique paths to /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_opportunities_unique_paths.csv
  - Paths with direct opportunity: 17
  - Paths without direct opportunity: 0
  - New column added: path_with_marker (marks direct_start_node with *asterisks*)


In [98]:
# Display examples of the new path_with_marker column
print("Examples of paths with direct_start_node marked with *asterisks*:\n")
for idx in range(min(5, len(direct_opportunities_df))):
    row = direct_opportunities_df.iloc[idx]
    print(f"Path {row['path_index']}: {row['path_pair_label']}")
    print(f"  Full path: {row['path_with_marker']}")
    print(f"  Excess nodes: {row['excess_nodes']}, Frames wasted: {row['frames_wasted']}")
    print()

Examples of paths with direct_start_node marked with *asterisks*:

Path 0: ('Target5', 'Target6')
  Full path: L521 → L410 → L520 → L410 → L35 → L22 → L34 → L49 → L518 → L49 → L518 → L49 → L518 → L49 → *L519* → L49 → L34 → L22 → L11 → L23 → L36 → L412 → L525
  Excess nodes: 14.0, Frames wasted: 179

Path 1: ('Target1', 'Target8')
  Full path: R522 → R411 → R35 → R22 → R34 → R49 → *R519* → R49 → R34 → R22 → R11 → R0 → R10 → R20 → R31 → R42 → R55
  Excess nodes: 6.0, Frames wasted: 142

Path 2: ('Target2', 'Target1')
  Full path: R526 → R413 → R36 → R23 → R11 → R22 → R35 → R410 → R520 → R410 → R35 → R22 → R34 → R49 → *R518* → R49 → R34 → R22 → R35 → R411 → R522
  Excess nodes: 12.0, Frames wasted: 155

Path 3: ('Target5', 'Target4')
  Full path: L521 → L410 → L35 → L22 → L11 → L23 → L36 → *L412* → L36 → L23 → L11 → L0 → L10 → L21 → L32 → L45 → L510
  Excess nodes: 6.0, Frames wasted: 459

Path 4: ('Target4', 'Target8')
  Full path: L510 → L45 → L32 → L44 → L58 → L44 → *L58* → L44 → L32 →

In [99]:
# Sanity check on the fix: every long indirect path must have a direct_start_node
# (worst case it's the destination node itself), and the marked node's remaining
# trajectory must equal the genuine shortest path from that node to the destination
# (the marked node itself need not lie on the destination's shortest path -- it's
# expected to be the apex of the last detour).
n_missing = direct_opportunities_df['direct_start_node'].isna().sum()
print(f"Paths missing a direct_start_node: {n_missing} (should be 0)")

bad_position = 0
for _, row in direct_opportunities_df.iterrows():
    seq = row['full_sequence']
    pos = row['direct_start_position_in_path']
    if pd.isna(pos):
        continue
    pos = int(pos)
    node, dest = seq[pos], seq[-1]
    sp = nx.shortest_path(G, node, dest)
    if seq[pos:] != sp:
        bad_position += 1
        print(f"Path {row['path_index']}: marked node {node} does not start a clean run to {dest}")

print(f"Paths with an inconsistent marker: {bad_position} (should be 0)")

Paths missing a direct_start_node: 0 (should be 0)
Paths with an inconsistent marker: 0 (should be 0)


# Cross-Session Indirect Path Analysis

For each of the 12 recording sessions, count all port-to-port trips where the animal visited **≥ 4 more nodes** than the graph-theoretic shortest path between the two ports ('long indirect' paths — same definition as the single-session analysis above).

**Metrics per session:**
- `n long indirect`: count of qualifying trips
- `% of trips`: fraction of all port-to-port segments
- `mean excess nodes`: mean node-count above shortest path (indirect trips only)
- `mean duration (s)`: mean total trip duration in seconds

Source: `node_summary_df.csv` (port-to-port path structure, identical methodology to cells above).

In [ ]:
from datetime import date as _date

BASE_DIR_ALL = Path('/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs')
FPS_GLOBAL   = 40

def _parse_session_date(folder_name):
    p = folder_name.split('_')
    return _date(int(p[2]), int(p[1]), int(p[0]))

per_session_rows = []
per_path_rows    = []

for d in sorted(BASE_DIR_ALL.iterdir()):
    if not d.is_dir() or d.name.startswith('plots_') or not d.name[0].isdigit():
        continue
    csv_path = d / 'node_summary_df.csv'
    if not csv_path.exists():
        print(f'  Skipping {d.name} — no node_summary_df.csv')
        continue

    node_df = pd.read_csv(csv_path).sort_values('node_visit_idx').reset_index(drop=True)

    # One row per port-to-port path segment (same logic as the single-session analysis above)
    path_segs, prev_label = [], None
    for _, row in node_df.iterrows():
        label = row['path_pair_label']
        if pd.isna(label):
            prev_label = None
            continue
        if label != prev_label:
            path_segs.append(row)
        prev_label = label

    total_segs    = len(path_segs)
    session_paths = []

    for row in path_segs:
        if row['port_to_port_path_type'] != 'indirect':
            continue
        excess = int(row['path_seq_length_unique']) - row['path_shortest_length']
        if excess < 4:
            continue
        # Duration from node_summary_df frame data
        start_nv = int(row['node_visit_idx'])
        seq_len  = int(row['path_seq_length_unique'])
        seg_rows = node_df[(node_df['node_visit_idx'] >= start_nv) &
                           (node_df['node_visit_idx'] <= start_nv + seq_len)]
        duration_sec = seg_rows['duration_frames'].sum() / FPS_GLOBAL
        entry = {
            'session':      d.name,
            'excess_nodes': excess,
            'path_pair_label': str(row['path_pair_label']),
            'duration_sec': duration_sec,
        }
        session_paths.append(entry)
        per_path_rows.append(entry)

    n = len(session_paths)
    per_session_rows.append({
        'session':           d.name,
        'date':              _parse_session_date(d.name),
        'n_segs_total':      total_segs,
        'n_indirect':        n,
        'pct_indirect':      100.0 * n / total_segs if total_segs else 0,
        'mean_excess_nodes': np.mean([e['excess_nodes'] for e in session_paths]) if n else np.nan,
        'mean_duration_sec': np.mean([e['duration_sec'] for e in session_paths]) if n else np.nan,
    })
    if n:
        print(f"  {d.name}: {n:3d}/{total_segs} long indirect  "
              f"mean_excess={np.mean([e['excess_nodes'] for e in session_paths]):.1f}  "
              f"mean_dur={np.mean([e['duration_sec'] for e in session_paths]):.1f}s")
    else:
        print(f'  {d.name}: 0 long indirect')

cross_session_df = (
    pd.DataFrame(per_session_rows)
    .sort_values('date')
    .reset_index(drop=True)
)
cross_session_df['day_label'] = cross_session_df['session'].str.replace('_2025', '', regex=False)

all_indirect_df = pd.DataFrame(per_path_rows)
_ord_map = {s: i for i, s in enumerate(cross_session_df['session'])}
all_indirect_df['_ord'] = all_indirect_df['session'].map(_ord_map)
all_indirect_df = all_indirect_df.sort_values('_ord').drop(columns='_ord').reset_index(drop=True)

print(f'\n{len(cross_session_df)} sessions, {len(all_indirect_df)} total long indirect paths')
display(cross_session_df[['day_label','n_segs_total','n_indirect','pct_indirect',
                           'mean_excess_nodes','mean_duration_sec']].rename(columns={
    'day_label':'session', 'n_segs_total':'total trips',
    'n_indirect':'n long indirect (≥4)', 'pct_indirect':'% of trips',
    'mean_excess_nodes':'mean excess nodes', 'mean_duration_sec':'mean dur (s)'}))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Indirect Path Behaviour Across Sessions  (excess ≥ 4 nodes above shortest)',
             fontweight='bold', fontsize=13)

_x  = np.arange(len(cross_session_df))
_xl = cross_session_df['day_label'].tolist()
_colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(_x)))

def _style_ax(ax, title, ylabel):
    ax.set_xticks(_x)
    ax.set_xticklabels(_xl, rotation=45, ha='right', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.3)

def _add_trend(ax, yvals):
    z = np.polyfit(_x, yvals, 1)
    ax.plot(_x, np.poly1d(z)(_x), '--', color='crimson', linewidth=1.5,
            label=f'trend ({z[0]:+.2f}/session)')
    ax.legend(fontsize=8)

# 1) Count of indirect paths
ax = axes[0, 0]
ax.bar(_x, cross_session_df['n_indirect'], color=_colors, edgecolor='black', linewidth=0.5)
ax.plot(_x, cross_session_df['n_indirect'], 'o-', color='black', linewidth=1.5, markersize=5, zorder=5)
_add_trend(ax, cross_session_df['n_indirect'])
_style_ax(ax, 'Number of Long Indirect Paths', 'Count')

# 2) % of valid trips
ax = axes[0, 1]
ax.bar(_x, cross_session_df['pct_indirect'], color=_colors, edgecolor='black', linewidth=0.5)
ax.plot(_x, cross_session_df['pct_indirect'], 'o-', color='black', linewidth=1.5, markersize=5, zorder=5)
_add_trend(ax, cross_session_df['pct_indirect'])
_style_ax(ax, '% of Trips That Are Long Indirect', '% of valid trips')

# 3) Mean excess nodes (among indirect paths)
ax = axes[1, 0]
ax.plot(_x, cross_session_df['mean_excess_nodes'], 'o-', color='mediumpurple', linewidth=2, markersize=7)
ax.fill_between(_x, cross_session_df['mean_excess_nodes'], alpha=0.15, color='mediumpurple')
_add_trend(ax, cross_session_df['mean_excess_nodes'])
_style_ax(ax, 'Mean Excess Nodes (indirect paths only)', 'Nodes above shortest path')

# 4) Mean duration of indirect paths
ax = axes[1, 1]
ax.plot(_x, cross_session_df['mean_duration_sec'], 'o-', color='teal', linewidth=2, markersize=7)
ax.fill_between(_x, cross_session_df['mean_duration_sec'], alpha=0.15, color='teal')
_add_trend(ax, cross_session_df['mean_duration_sec'])
_style_ax(ax, 'Mean Duration of Indirect Paths', 'Seconds')

plt.tight_layout()
plt.savefig(BASE_DIR_ALL / 'indirect_path_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: indirect_path_learning_curves.png')

In [ ]:
import seaborn as sns

_session_order = cross_session_df['session'].tolist()
_day_labels    = cross_session_df['day_label'].tolist()
_palette       = dict(zip(_session_order, plt.cm.viridis(
    __import__('numpy').linspace(0.15, 0.85, len(_session_order)))))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Per-Path Distributions Across Sessions  (indirect paths ≥ 4 excess nodes)',
             fontweight='bold', fontsize=12)

for ax, col, title, ylabel in [
    (axes[0], 'excess_nodes', 'Excess Nodes Distribution', 'Nodes above shortest path'),
    (axes[1], 'duration_sec', 'Path Duration Distribution', 'Duration (seconds)'),
]:
    sns.violinplot(data=all_indirect_df, x='session', y=col,
                   order=_session_order, hue='session', palette=_palette,
                   legend=False, ax=ax, cut=0, inner='quartile', linewidth=0.8)
    sns.stripplot(data=all_indirect_df, x='session', y=col,
                  order=_session_order, color='black', alpha=0.35, size=3, ax=ax, jitter=True)
    ax.set_xticks(range(len(_session_order)))
    ax.set_xticklabels(_day_labels, rotation=45, ha='right', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR_ALL / 'indirect_path_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: indirect_path_distributions.png')

# Spike Data

In [101]:
good_units = cluster_info[cluster_info['group'] == 'good'].copy().reset_index(drop=True)
mua_units  = cluster_info[cluster_info['group'] == 'mua'].copy().reset_index(drop=True)
# noise units are dropped entirely

print(f"Total clusters : {len(cluster_info)}")
print(f"Good units     : {len(good_units)}")
print(f"MUA units      : {len(mua_units)}")
print(f"Noise dropped  : {(cluster_info['group'] == 'noise').sum()}")

Total clusters : 1005
Good units     : 34
MUA units      : 137
Noise dropped  : 793


In [ ]:
FS = 30_000  # Neuropixels sampling rate (Hz)

# spike_times / spike_clusters are dicts from scipy.io.loadmat;
# extract the underlying numpy object array (one element per trial)
def _unwrap_mat(d):
    keys = [k for k in d if not k.startswith('__')]
    arr = d[keys[0]]
    return arr.flatten()  # handles (1,N) or (N,1) cell arrays

spike_times_all    = _unwrap_mat(spike_times)
spike_clusters_all = _unwrap_mat(spike_clusters)

# Maze trials are 31–766 (1-indexed) → indices 30:767 (0-indexed, 737 trials)
MAZE_START = 30
MAZE_END   = 767

maze_spike_times    = spike_times_all[MAZE_START:MAZE_END]
maze_spike_clusters = spike_clusters_all[MAZE_START:MAZE_END]

assert len(maze_spike_times) == 737, f"Expected 737 maze trials, got {len(maze_spike_times)}"
print(f"Maze trials extracted: {len(maze_spike_times)} (trials 31–766)")
print()

# Validate: print spike count and trial length (in samples and seconds) for every trial
# Spike times within each trial are assumed to be relative to trial start (in samples)
trial_lengths_s = []
print(f"{'Trial':>7}  {'Spikes':>7}  {'Length (samples)':>17}  {'Length (s)':>11}")
print("-" * 48)
for i, st in enumerate(maze_spike_times):
    st_flat = np.asarray(st).flatten()
    n_spikes = len(st_flat)
    length_samples = int(st_flat.max() - st_flat.min()) if n_spikes > 1 else 0
    length_s = length_samples / FS
    trial_lengths_s.append(length_s)
    print(f"{MAZE_START + i + 1:>7}  {n_spikes:>7}  {length_samples:>17}  {length_s:>11.4f}")

print()
trial_lengths_s = np.array(trial_lengths_s)
print(f"Summary — length (s):  min={trial_lengths_s.min():.2f}  max={trial_lengths_s.max():.2f}  mean={trial_lengths_s.mean():.2f}")


# Change-of-mind events
One event per long indirect path. Time 0 is the moment the mouse exits the apex node (marked `direct_start_node`) and commits to the direct route. The PSTH window is *not* a fixed ±2s — trial-switching mid-path is irrelevant to the question being asked, so the window spans the whole indirect path: from the moment the mouse leaves the source port to the moment it arrives at the destination port, regardless of how many of the pipeline's fine-grained "trials" that spans.

In [106]:
import ast

# summary_df has MultiIndex columns (from the tracking pipeline); flatten the
# behavior-only columns we need.
sdf = summary_df.copy()
sdf.columns = ['_'.join([c for c in col if c]) if isinstance(col, tuple) else col for col in sdf.columns]

# Per-trial start/end on the GLOBAL video clock (frame_idx_global / FPS is already a
# continuous, session-wide clock, so no per-trial offsetting is needed on the video side).
trial_meta = sdf.groupby('trial_idx').agg(
    start_frame=('frame_idx_global', 'min'),
    trial_length=('trial_length', 'first'),
)
trial_meta['start_abs'] = trial_meta['start_frame'] / FPS
trial_meta['end_abs'] = trial_meta['start_abs'] + trial_meta['trial_length']

def trials_overlapping(t_lo, t_hi):
    """All pipeline 'trial' indices whose time range overlaps [t_lo, t_hi].

    We deliberately don't derive this from the trial_idx values seen on the path's own
    node rows: a trial with no recorded node visit (e.g. very short) would silently drop
    out and create a gap in the middle of the spike window. Going by absolute time overlap
    instead guarantees full coverage of the path's duration.
    """
    m = (trial_meta['end_abs'] >= t_lo) & (trial_meta['start_abs'] <= t_hi)
    return trial_meta.index[m].tolist()

def parse_pair_label(label):
    try:
        return ast.literal_eval(label)
    except (ValueError, SyntaxError, TypeError):
        return ('NA', 'NA')

change_of_mind_events = []
for _, row in direct_opportunities_df.iterrows():
    apex_visit_idx = row['direct_start_node_visit_idx']
    if pd.isna(apex_visit_idx):
        continue

    start_nv = row['start_node_visit_idx']
    end_nv = start_nv + row['path_seq_length_unique']
    path_rows = improved_node_summary_df[
        (improved_node_summary_df['node_visit_idx'] >= start_nv) &
        (improved_node_summary_df['node_visit_idx'] <= end_nv)
    ]
    apex_row = path_rows[path_rows['node_visit_idx'] == int(apex_visit_idx)].iloc[0]

    path_start_abs = path_rows.iloc[0]['start_frame'] / FPS   # mouse leaves the source port
    path_end_abs = path_rows.iloc[-1]['start_frame'] / FPS    # mouse arrives at the destination port
    t0_abs = apex_row['end_frame'] / FPS                       # mouse leaves the apex node

    trials_in_path = trials_overlapping(path_start_abs, path_end_abs)
    from_port, to_port = parse_pair_label(row['path_pair_label'])

    # One (node_name, time-relative-to-apex-exit) pair per node visit in the path, for
    # annotating the PSTH x-axis with the actual port-to-port trajectory.
    node_track = [
        (r['node_name'], r['start_frame'] / FPS - t0_abs)
        for _, r in path_rows.iterrows()
    ]

    change_of_mind_events.append({
        'path_index': int(row['path_index']),
        'from_port': from_port,
        'to_port': to_port,
        'apex_node': row['direct_start_node'],
        'trials_in_path': trials_in_path,
        't0_abs': t0_abs,
        'window_pre': t0_abs - path_start_abs,    # seconds of path data before the apex exit
        'window_post': path_end_abs - t0_abs,     # seconds of path data after the apex exit
        'node_track': node_track,
    })

change_of_mind_events_df = pd.DataFrame(change_of_mind_events)
print(f"Built {len(change_of_mind_events_df)} change-of-mind events (one per long indirect path)")
display(change_of_mind_events_df)

Built 17 change-of-mind events (one per long indirect path)


,path_index,from_port,to_port,apex_node,trials_in_path,t0_abs,window_pre,window_post,node_track
0,0,Target5,Target6,L519,"[7, 8, 9]",56.975,17.125,3.400,"[(L521, -17.125), (L410, -9.25), (L520, -9.100..."
1,1,Target1,Target8,R519,"[31, 32]",151.250,1.600,3.525,"[(R522, -1.5999999999999943), (R411, -1.5), (R..."
2,2,Target2,Target1,R518,"[166, 167]",656.150,8.000,2.100,"[(R526, -8.0), (R413, -7.625), (R36, -7.0), (R..."
3,3,Target5,Target4,L412,"[218, 219, 220]",896.800,9.750,11.100,"[(L521, -9.75), (L410, -3.25), (L35, -2.949999..."
4,4,Target4,Target8,L58,[221],920.225,3.675,6.025,"[(L510, -3.675000000000068), (L45, -2.55000000..."
5,5,Target1,Target2,R34,"[372, 373]",1574.075,1.900,6.575,"[(R522, -1.900000000000091), (R411, -1.75), (R..."
6,6,Target3,Target4,L51,"[377, 378]",1626.800,13.200,3.750,"[(L56, -13.200000000000045), (L43, -12.7999999..."
7,7,Target1,Target2,R410,"[402, 403]",1739.200,3.000,4.475,"[(R522, -3.0), (R411, -2.75), (R35, -2.0250000..."
8,8,Target6,Target5,L529,"[516, 517, 518]",2239.525,9.575,2.850,"[(L525, -9.575000000000273), (L412, -2.8499999..."
9,9,Target2,Target4,L512,"[630, 631]",2926.525,36.550,3.600,"[(R526, -36.55000000000018), (R413, -34.425000..."


# PSTH plots: good units aligned to the change-of-mind moment
One PSTH per (indirect path × good unit), saved under `resources/outputs/{session}/psth/path_{idx}_{from}_to_{to}/unit_{cluster_id}.png`. Time 0 = the moment the mouse exits the apex node for that path; the window spans the *entire* indirect path (source-port departure to destination-port arrival), stitching spikes across trial boundaries as needed. The firing rate is shown as a Gaussian-smoothed continuous line, with the raw binned rate underneath as a light reference.

In [110]:
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

BIN_SIZE = 0.05      # seconds, histogram bin for the raw rate
SMOOTH_SIGMA = 0.15  # seconds, stdev of the Gaussian smoothing kernel
sigma_bins = SMOOTH_SIGMA / BIN_SIZE

def gather_relative_spikes(trials_in_path, cluster_id, t0_abs):
    """Spikes for one cluster across all trials touched by the path, expressed as
    seconds relative to t0_abs (the change-of-mind moment), by stitching each trial's
    spike-times (stored relative to that trial's own start, in samples) onto the
    global video clock via trial_meta['start_abs'].
    """
    rel_chunks = []
    for t in trials_in_path:
        spike_t = np.asarray(maze_spike_times[t]).flatten() / FS
        spike_c = np.asarray(maze_spike_clusters[t]).flatten()
        abs_t = trial_meta.loc[t, 'start_abs'] + spike_t
        rel_chunks.append(abs_t[spike_c == cluster_id] - t0_abs)
    return np.concatenate(rel_chunks) if rel_chunks else np.array([])

def binned_and_smoothed_rate(rel_t, bin_edges):
    """Raw per-bin firing rate (Hz) and its Gaussian-smoothed counterpart."""
    counts, _ = np.histogram(rel_t, bins=bin_edges)
    rate = counts.astype(float) / BIN_SIZE
    smoothed = gaussian_filter1d(rate, sigma=sigma_bins, mode='constant')
    return rate, smoothed

psth_out_dir = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/psth")
psth_out_dir.mkdir(parents=True, exist_ok=True)

n_events = len(change_of_mind_events_df)
n_units = len(good_units)
print(f"Generating {n_events} x {n_units} = {n_events * n_units} PSTH plots into {psth_out_dir}")

n_saved = 0
for _, event in change_of_mind_events_df.iterrows():
    window_pre = event['window_pre']
    window_post = event['window_post']
    n_bins = max(1, round((window_pre + window_post) / BIN_SIZE))
    bin_edges = np.linspace(-window_pre, window_post, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    event_dir = psth_out_dir / f"path_{event['path_index']:02d}_{event['from_port']}_to_{event['to_port']}"
    event_dir.mkdir(parents=True, exist_ok=True)

    for _, unit in good_units.iterrows():
        cluster_id = int(unit['cluster_id'])
        rel_t = gather_relative_spikes(event['trials_in_path'], cluster_id, event['t0_abs'])
        rate, smoothed_rate = binned_and_smoothed_rate(rel_t, bin_edges)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(bin_centers, rate, width=BIN_SIZE, align='center', color='lightsteelblue', alpha=0.5, label='raw (binned)')
        ax.plot(bin_centers, smoothed_rate, color='steelblue', linewidth=1.8, label='smoothed')
        ax.axvline(0, color='red', linestyle='--', linewidth=1, label='change of mind (exit apex)')
        ax.set_xlabel('Time relative to change-of-mind (s)')
        ax.set_ylabel('Firing rate (Hz)')
        ax.set_title(f"Unit {cluster_id} | path {event['path_index']}: {event['from_port']}→{event['to_port']} | apex={event['apex_node']}", fontsize=9)
        ax.set_xlim(-window_pre, window_post)
        ax.legend(fontsize=7, loc='upper right')
        fig.tight_layout()

        fig.savefig(event_dir / f"unit_{cluster_id:04d}_good.png", dpi=120)
        plt.close(fig)
        n_saved += 1

print(f"Saved {n_saved} PSTH plots under {psth_out_dir}")

Generating 17 x 34 = 578 PSTH plots into /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/psth
Saved 578 PSTH plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/psth


# PSTH plots annotated with the path trajectory
Same smoothed PSTH, but with a secondary x-axis below the time axis showing which maze node the mouse was at, at each point along the path — so the port-to-port transitions are visible alongside the firing rate. Saved under `resources/outputs/{session}/psth_annotated/path_{idx}_{from}_to_{to}/unit_{cluster_id}.png`.

In [ ]:
psth_annotated_dir = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/psth_annotated")
psth_annotated_dir.mkdir(parents=True, exist_ok=True)

print(f"Generating {n_events} x {n_units} = {n_events * n_units} annotated PSTH plots into {psth_annotated_dir}")

n_saved = 0
for _, event in change_of_mind_events_df.iterrows():
    window_pre = event['window_pre']
    window_post = event['window_post']
    n_bins = max(1, round((window_pre + window_post) / BIN_SIZE))
    bin_edges = np.linspace(-window_pre, window_post, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    node_names = [name for name, _ in event['node_track']]
    node_times = [t for _, t in event['node_track']]

    event_dir = psth_annotated_dir / f"path_{event['path_index']:02d}_{event['from_port']}_to_{event['to_port']}"
    event_dir.mkdir(parents=True, exist_ok=True)

    # widen the figure a bit when the path visits many nodes, so the bottom labels stay legible
    fig_width = max(6, len(node_names) * 0.35)

    for _, unit in good_units.iterrows():
        cluster_id = int(unit['cluster_id'])
        rel_t = gather_relative_spikes(event['trials_in_path'], cluster_id, event['t0_abs'])
        rate, smoothed_rate = binned_and_smoothed_rate(rel_t, bin_edges)

        fig, ax = plt.subplots(figsize=(fig_width, 4))
        ax.bar(bin_centers, rate, width=BIN_SIZE, align='center', color='lightsteelblue', alpha=0.5, label='raw (binned)')
        ax.plot(bin_centers, smoothed_rate, color='steelblue', linewidth=1.8, label='smoothed')
        ax.axvline(0, color='red', linestyle='--', linewidth=1, label='change of mind (exit apex)')
        ax.set_ylabel('Firing rate (Hz)')
        ax.set_title(f"Unit {cluster_id} | path {event['path_index']}: {event['from_port']}→{event['to_port']} | apex={event['apex_node']}", fontsize=9)
        ax.set_xlim(-window_pre, window_post)
        ax.set_xlabel('Time relative to change-of-mind (s)')
        ax.legend(fontsize=7, loc='upper right')

        # secondary x-axis sharing the same time scale, ticked at each node visit in the path
        secax = ax.secondary_xaxis(-0.35)
        secax.set_xticks(node_times)
        secax.set_xticklabels(node_names, rotation=90, fontsize=6)
        secax.set_xlabel('Node visited', fontsize=8)

        fig.tight_layout()
        fig.savefig(event_dir / f"unit_{cluster_id:04d}_good.png", dpi=180)
        plt.close(fig)
        n_saved += 1

print(f"Saved {n_saved} annotated PSTH plots under {psth_annotated_dir}")

Generating 17 x 34 = 578 annotated PSTH plots into /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/psth_annotated


# Population-level analysis (PCA): good + MUA units around the change-of-mind moment
For each indirect path, build a time × units population firing-rate matrix (good + MUA units, z-scored per unit) over the path's full time window, and run PCA on it. Rather than just taking PC1/PC2 by variance, we score every PC (among the top-variance candidates) by how much its trajectory actually shifts right around t=0, and visualize the 2 most relevant ones as a 2D trajectory across the path.

In [ ]:
# Population = good + MUA units (noise still excluded)
population_units = pd.concat([good_units, mua_units], ignore_index=True)
print(f"Population analysis units: {len(population_units)} ({len(good_units)} good + {len(mua_units)} mua)")

# How many of the top-variance PCs to even consider as "relevant" candidates. Ranking
# purely by change-at-t0 score without this cap would let a late, noise-dominated PC
# win just by chance fluctuating near t=0; restricting to a variance-meaningful pool
# avoids that.
N_CANDIDATE_PCS = 20
T0_CHANGE_WINDOW = 0.5  # seconds on each side of t=0 used to score "change at the change-of-mind"

def build_population_rate_matrix(event, units_df):
    """Time x units matrix of (smoothed) firing rates over the path's full window."""
    window_pre = event['window_pre']
    window_post = event['window_post']
    n_bins = max(1, round((window_pre + window_post) / BIN_SIZE))
    bin_edges = np.linspace(-window_pre, window_post, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    cluster_ids = units_df['cluster_id'].astype(int).to_numpy()
    rate_matrix = np.zeros((n_bins, len(cluster_ids)))
    for j, cid in enumerate(cluster_ids):
        rel_t = gather_relative_spikes(event['trials_in_path'], cid, event['t0_abs'])
        _, smoothed = binned_and_smoothed_rate(rel_t, bin_edges)
        rate_matrix[:, j] = smoothed
    return bin_centers, rate_matrix, cluster_ids

def run_pca(rate_matrix):
    """PCA via SVD on the z-scored (per unit) rate matrix. Returns PC scores
    (time x components) and the fraction of variance each component explains."""
    mu = rate_matrix.mean(axis=0)
    sigma = rate_matrix.std(axis=0)
    sigma_safe = np.where(sigma > 1e-9, sigma, 1.0)  # silent units: leave as all-zero after centering
    X = (rate_matrix - mu) / sigma_safe

    U, S, _Vt = np.linalg.svd(X, full_matrices=False)
    scores = U * S
    explained_var = (S ** 2) / np.sum(S ** 2)
    return scores, explained_var

def pc_change_scores(scores, bin_centers, window_pre, window_post, t0_window=T0_CHANGE_WINDOW):
    """For each PC, |mean(post t0) - mean(pre t0)| normalized by that PC's own std --
    an effect-size for how much the trajectory shifts right at the change-of-mind moment."""
    w = min(t0_window, window_pre, window_post)
    pre_mask = (bin_centers >= -w) & (bin_centers < 0)
    post_mask = (bin_centers >= 0) & (bin_centers <= w)

    out = np.zeros(scores.shape[1])
    for k in range(scores.shape[1]):
        pc = scores[:, k]
        pre_mean = pc[pre_mask].mean() if pre_mask.any() else 0.0
        post_mean = pc[post_mask].mean() if post_mask.any() else 0.0
        pc_std = pc.std() if pc.std() > 1e-9 else 1.0
        out[k] = abs(post_mean - pre_mean) / pc_std
    return out

# Population PCA trajectory: top 3 PCs by variance
Alternative to the "best 2" (change-relevance-ranked) view above: just take PC1–PC3, the 3 highest-variance components, and plot the population trajectory in 3D. Same matrices/PCA as before (good + MUA units, z-scored, one PCA per path) — only the PC selection criterion and the plot dimensionality differ.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the '3d' projection)

pca_top3_dir = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/population_pca_top3pcs")
pca_top3_dir.mkdir(parents=True, exist_ok=True)

pca_top3_summary_rows = []

for _, event in change_of_mind_events_df.iterrows():
    bin_centers, rate_matrix, cluster_ids = build_population_rate_matrix(event, population_units)
    scores, explained_var = run_pca(rate_matrix)
    change = pc_change_scores(scores, bin_centers, event['window_pre'], event['window_post'])

    pc1, pc2, pc3 = 0, 1, 2  # fixed: first 3 PCs by variance, not change-relevance
    traj_x, traj_y, traj_z = scores[:, pc1], scores[:, pc2], scores[:, pc3]
    t0_idx = np.argmin(np.abs(bin_centers))

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(traj_x, traj_y, traj_z, color='gray', alpha=0.4, linewidth=1, zorder=1)
    sca = ax.scatter(traj_x, traj_y, traj_z, c=bin_centers, cmap='viridis', s=18, zorder=2)
    ax.scatter(*[a[0] for a in (traj_x, traj_y, traj_z)], color='green', marker='o', s=70, zorder=3, label='path start')
    ax.scatter(*[a[-1] for a in (traj_x, traj_y, traj_z)], color='black', marker='s', s=70, zorder=3, label='path end')
    ax.scatter(*[a[t0_idx] for a in (traj_x, traj_y, traj_z)], color='red', marker='*', s=280, zorder=4, label='change of mind')

    fig.colorbar(sca, ax=ax, label='Time relative to change-of-mind (s)', shrink=0.6, pad=0.1)
    ax.set_xlabel(f"PC1 ({explained_var[pc1] * 100:.1f}% var)")
    ax.set_ylabel(f"PC2 ({explained_var[pc2] * 100:.1f}% var)")
    ax.set_zlabel(f"PC3 ({explained_var[pc3] * 100:.1f}% var)")
    ax.set_title(f"Population trajectory (top 3 PCs) | path {event['path_index']}: {event['from_port']}→{event['to_port']} | apex={event['apex_node']}", fontsize=9)
    ax.legend(fontsize=8)
    fig.tight_layout()

    out_path = pca_top3_dir / f"path_{event['path_index']:02d}_{event['from_port']}_to_{event['to_port']}.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

    pca_top3_summary_rows.append({
        'path_index': event['path_index'],
        'from_port': event['from_port'],
        'to_port': event['to_port'],
        'n_units': len(cluster_ids),
        'n_bins': len(bin_centers),
        'pc1_explained_var': explained_var[pc1],
        'pc1_change_score': change[pc1],
        'pc2_explained_var': explained_var[pc2],
        'pc2_change_score': change[pc2],
        'pc3_explained_var': explained_var[pc3],
        'pc3_change_score': change[pc3],
    })

pca_top3_summary_df = pd.DataFrame(pca_top3_summary_rows)
pca_top3_summary_df.to_csv(pca_top3_dir / 'pca_top3_summary.csv', index=False)
print(f"Saved {len(pca_top3_summary_df)} top-3-PC population trajectory plots + summary under {pca_top3_dir}")
display(pca_top3_summary_df)

# Videos: Indirect Paths with Change-of-Mind Annotation

For each of the 17 long indirect paths, create a side-by-side video (raw video + maze graph) with the change-of-mind moment annotated:
- **Graph**: the apex node is always highlighted with a crosshair — orange/amber before the event, switching to red at the change-of-mind frame.
- **Video**: a bold "CHANGE OF MIND" banner appears at the apex-exit frame and remains on screen for the rest of the clip.
- **Flash**: a red border fades out over ~2 seconds immediately after the event.

Videos are saved under `resources/outputs/{session}/indirect_path_videos/`.

In [21]:
from pathlib import Path
import pandas as pd

# Video file (same naming convention as align_frames_trials.ipynb)
maze_video_path = rf"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/maze_videos/NPC1_{YYYY}_{MM}_{DD}.mp4"
print(f"Video path : {maze_video_path}")
print(f"Video exists: {Path(maze_video_path).exists()}")

# Port label screen positions (pixel coords in the original 1280×960 video)
port_label_positions = pd.DataFrame([
    {'port_label': 'Target1', 'x': 945, 'y':  45},
    {'port_label': 'Target2', 'x': 700, 'y':  45},
    {'port_label': 'Target3', 'x': 350, 'y':  45},
    {'port_label': 'Target4', 'x': 120, 'y':  45},
    {'port_label': 'Target5', 'x': 120, 'y': 890},
    {'port_label': 'Target6', 'x': 360, 'y': 890},
    {'port_label': 'Target7', 'x': 945, 'y': 890},
    {'port_label': 'Target8', 'x': 695, 'y': 890},
])

# Reward leaf nodes and their parents (used for reward icon placement on the video)
if MM == '09' and DD in ['08', '09', '10', '11']:
    reward_leaf_parents = {
        1: ('R522', ['R411']),
        2: ('R526', ['R413']),
        3: ('L56',  ['L43']),
        4: ('L510', ['L45']),
        5: ('L521', ['L410']),
        6: ('L525', ['L412']),
        7: ('R59',  ['R44']),
        8: ('R55',  ['R42']),
    }
else:
    reward_leaf_parents = {
        1: ('R522', ['R411']),
        2: ('R526', ['R413']),
        3: ('L56',  ['L43']),
        4: ('L510', ['L45']),
        5: ('L521', ['L410']),
        6: ('L525', ['L412']),
        7: ('R55',  ['R42']),
        8: ('R59',  ['R44']),
    }

Video path : /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/maze_videos/NPC1_2025_09_10.mp4
Video exists: True


In [22]:
def create_indirect_path_video(start_frame, end_frame, output_path, summary_df, G, node_positions, port_map,
                               video_path, change_of_mind_frame=None, apex_node_id=None,
                               reward_leaf_parents=None, fps=40, video_height=720,
                               graph_width=400, bodypart_colors=None,
                               port_label_positions=None, port_colors=None):
    """
    Side-by-side video (raw video + maze graph) for one indirect path, with the
    change-of-mind moment annotated:
      - Apex node shown on the graph with a crosshair (orange before event, red after).
      - 'CHANGE OF MIND' banner appears on the video panel from change_of_mind_frame onward.
      - Red border flash fades out over ~2 s immediately after the event.
    Based on create_side_by_side_video from align_frames_trials.ipynb.
    """
    import cv2
    import numpy as np
    import pandas as pd
    from pathlib import Path

    video_path = Path(video_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    reward_size_to_px = {}

    if reward_leaf_parents is None:
        reward_leaf_parents = {}

    def _resolve_column(df, candidates):
        cols = df.columns
        candidate_l = [c.lower() for c in candidates]
        if isinstance(cols, pd.MultiIndex):
            for col in cols:
                parts = [str(v).strip().lower() for v in col]
                for candidate in candidate_l:
                    if candidate in parts:
                        return col
            return None
        lower_map = {str(col).strip().lower(): col for col in cols}
        for candidate in candidate_l:
            if candidate in lower_map:
                return lower_map[candidate]
        return None

    def _resolve_tracking_column(df, bodypart, field):
        cols = df.columns
        if isinstance(cols, pd.MultiIndex):
            for col in cols:
                if len(col) >= 2 and str(col[0]) == bodypart and str(col[1]).lower() == field.lower():
                    return col
            return None
        for col in [f'{bodypart}_{field}', f'{bodypart}.{field}', f'{bodypart}{field}']:
            if col in cols:
                return col
        return None

    def _normalize_node(value):
        if value is None or pd.isna(value):
            return None
        text = str(value).strip().strip('"').strip("'")
        if text.lower() in {'none', 'nan', ''}:
            return None
        return text if text in node_positions else None

    def _parse_edge_value(value):
        if value is None or pd.isna(value):
            return None
        if isinstance(value, (list, tuple)) and len(value) == 2:
            src = _normalize_node(value[0])
            dst = _normalize_node(value[1])
            if src is not None and dst is not None:
                return (src, dst)
            return None
        text = str(value)
        if text.lower().strip() in {'none', 'nan', ''}:
            return None
        hits = [node for node in node_positions.keys() if node in text]
        if len(hits) >= 2:
            return (hits[0], hits[1])
        for sep in ['->', '-', '|', ',', ' to ']:
            if sep in text:
                parts = [p.strip().strip("()[]\"'") for p in text.split(sep)]
                parts = [p for p in parts if p in node_positions]
                if len(parts) >= 2:
                    return (parts[0], parts[1])
        return None

    def _normalize_port_label_positions(value):
        positions = {}
        if value is None:
            return positions
        if isinstance(value, pd.DataFrame):
            if {'port_label', 'x', 'y'}.issubset(value.columns):
                for _, row in value.iterrows():
                    label = str(row['port_label']).strip()
                    if label:
                        positions[label] = (float(row['x']), float(row['y']))
            return positions
        if isinstance(value, dict):
            for label, pos in value.items():
                label = str(label).strip()
                if isinstance(pos, (list, tuple)) and len(pos) >= 2:
                    positions[label] = (float(pos[0]), float(pos[1]))
            return positions
        return positions

    def hex_to_bgr(h):
        h = h.lstrip('#')
        lv = len(h)
        rgb = tuple(int(h[i:i + lv // 3], 16) for i in range(0, lv, lv // 3))
        return (int(rgb[2]), int(rgb[1]), int(rgb[0]))

    icon_path = Path('/var/home/almogmeir/Documents/M.Sc/Project/Input/water-drop-icon.png')
    icon_drop = cv2.imread(str(icon_path), cv2.IMREAD_UNCHANGED) if icon_path.exists() else None

    icon_x_path = Path('/var/home/almogmeir/Documents/M.Sc/Project/Input/red_x_icon.png')
    if icon_x_path.exists():
        icon_x = cv2.imread(str(icon_x_path), cv2.IMREAD_UNCHANGED)
    else:
        size_px = 26
        icon_x = np.zeros((size_px, size_px, 4), dtype=np.uint8)
        t = max(4, size_px // 20)
        c = (0, 0, 255, 255)
        cv2.line(icon_x, (int(size_px*0.15), int(size_px*0.15)), (int(size_px*0.85), int(size_px*0.85)), c, t, cv2.LINE_AA)
        cv2.line(icon_x, (int(size_px*0.85), int(size_px*0.15)), (int(size_px*0.15), int(size_px*0.85)), c, t, cv2.LINE_AA)

    def _overlay_icon(img, center, icon_img):
        if icon_img is None:
            return
        h, w = icon_img.shape[:2]
        cx, cy = int(center[0]), int(center[1])
        x1, y1 = max(0, cx - w // 2), max(0, cy - h // 2)
        x2, y2 = min(img.shape[1], cx + w // 2), min(img.shape[0], cy + h // 2)
        rw, rh = x2 - x1, y2 - y1
        if rw <= 0 or rh <= 0:
            return
        ix1 = 0 if cx - w // 2 >= 0 else -(cx - w // 2)
        iy1 = 0 if cy - h // 2 >= 0 else -(cy - h // 2)
        roi = img[y1:y2, x1:x2].astype(float)
        if icon_img.shape[2] == 4:
            bgr = icon_img[:, :, :3].astype(float)
            alpha = icon_img[:, :, 3].astype(float) / 255.0
        else:
            bgr = icon_img.astype(float)
            alpha = np.ones((h, w), dtype=float)
        ic = bgr[iy1:iy1+rh, ix1:ix1+rw]
        ac = alpha[iy1:iy1+rh, ix1:ix1+rw]
        for ch in range(3):
            roi[:, :, ch] = ic[:, :, ch] * ac + roi[:, :, ch] * (1 - ac)
        img[y1:y2, x1:x2] = roi.astype(np.uint8)

    reward_size_col = _resolve_column(summary_df, ['reward_size'])
    if reward_size_col is not None:
        vals = summary_df[reward_size_col]
        non_zero_vals = sorted(pd.unique(vals.dropna().astype(float)))
        non_zero_vals = [v for v in non_zero_vals if v != 0]
        size_pixels = [12, 16, 20, 26]
        for i, v in enumerate(non_zero_vals[:4]):
            reward_size_to_px[float(v)] = size_pixels[i]
        if len(non_zero_vals) > 4:
            for v in non_zero_vals[4:]:
                reward_size_to_px[float(v)] = size_pixels[-1]

    icon_drop_resized = {}
    sizes_to_resize = set(reward_size_to_px.values()) | {26}
    for px_size in sizes_to_resize:
        if icon_drop is not None:
            icon_drop_resized[int(px_size)] = cv2.resize(icon_drop, (int(px_size), int(px_size)), interpolation=cv2.INTER_LINEAR)

    icon_x_resized = cv2.resize(icon_x, (24, 24), interpolation=cv2.INTER_LINEAR) if icon_x is not None else None

    if bodypart_colors is None:
        bodypart_colors = {
            'headstage': hex_to_bgr('#ff595e'), 'ear_L': hex_to_bgr('#ffca3a'),
            'ear_R': hex_to_bgr('#8ac926'), 'midbody': hex_to_bgr('#1982c4'),
            'tailbase': hex_to_bgr('#6a4c93'),
        }

    if port_colors is None:
        port_colors = {
            'Target1': (178, 140, 87), 'Target2': (122, 165, 110), 'Target3': (143, 108, 183),
            'Target4': (104, 170, 196), 'Target5': (171, 122, 164), 'Target6': (96, 147, 186),
            'Target7': (153, 151, 102), 'Target8': (128, 112, 184),
        }

    port_label_positions = _normalize_port_label_positions(port_label_positions)

    bodypart_columns = {}
    for bp in ['headstage', 'ear_L', 'ear_R', 'midbody', 'tailbase']:
        x_col = _resolve_tracking_column(summary_df, bp, 'x')
        y_col = _resolve_tracking_column(summary_df, bp, 'y')
        if x_col is not None and y_col is not None:
            bodypart_columns[bp] = {'x': x_col, 'y': y_col}

    node_col = _resolve_column(summary_df, ['headstage_graph_node'])
    edge_col = _resolve_column(summary_df, ['headstage_graph_edge'])
    frame_map_col = _resolve_column(summary_df, ['frame_idx_global', 'frame_idx'])
    reward_port_col = _resolve_column(summary_df, ['reward_port', 'target_port'])

    print(f"Found {len(bodypart_columns)} bodyparts, reward_size_col={reward_size_col}")

    cap = cv2.VideoCapture(str(video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    original_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    video_width = int(video_height * original_width / original_height)
    total_width = video_width + graph_width
    total_height = video_height

    if frame_map_col is not None:
        _frame_np = pd.to_numeric(summary_df[frame_map_col], errors='coerce').to_numpy()
        _match_ilocs = np.where((_frame_np >= float(start_frame)) & (_frame_np <= float(end_frame)))[0]
        if len(_match_ilocs) == 0:
            valid = _frame_np[~np.isnan(_frame_np)]
            print(f"Warning: no rows with frame_idx_global in [{start_frame}, {end_frame}]. "
                  f"summary_df range: [{int(valid.min())}, {int(valid.max())}]")
            cap.release()
            return
        _start_iloc, _end_iloc = int(_match_ilocs[0]), int(_match_ilocs[-1])
    else:
        print("Warning: frame_idx_global column not found — falling back to row-index interpretation.")
        _start_iloc = max(0, int(start_frame))
        _end_iloc = min(int(end_frame), len(summary_df) - 1)
    n_frames = _end_iloc - _start_iloc + 1

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (total_width, total_height))

    if node_positions:
        all_x = [pos[0] for pos in node_positions.values()]
        all_y = [pos[1] for pos in node_positions.values()]
        min_x, max_x = min(all_x), max(all_x)
        min_y, max_y = min(all_y), max(all_y)
        data_width = max_x - min_x if max_x > min_x else 1
        data_height = max_y - min_y if max_y > min_y else 1
        padding = 30
        scale = min((graph_width - 2*padding) / data_width, (total_height - 2*padding) / data_height, 1.0)
        scaled_width = data_width * scale
        scaled_height = data_height * scale
        offset_x = video_width + padding + ((graph_width - 2*padding) - scaled_width) / 2
        offset_y = padding + ((total_height - 2*padding) - scaled_height) / 2
    else:
        scale = 1.0
        offset_x = video_width + 30
        offset_y = 30
        min_x = min_y = 0

    graph_xy = {
        nid: (int((pos[0] - min_x) * scale + offset_x), int((pos[1] - min_y) * scale + offset_y))
        for nid, pos in node_positions.items()
    }

    print(f"Creating {n_frames} frames...")

    reward_icon_timers = {}
    current_color = hex_to_bgr("#975100")

    for row_iloc in range(_start_iloc, _end_iloc + 1):
        row = summary_df.iloc[row_iloc]
        if frame_map_col is not None and pd.notna(row[frame_map_col]):
            video_frame_idx = int(row[frame_map_col])
        else:
            video_frame_idx = row_iloc
        video_frame_idx = max(0, min(video_frame_idx, frame_count - 1))

        cap.set(cv2.CAP_PROP_POS_FRAMES, video_frame_idx)
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (video_width, video_height))
        output_frame = np.full((total_height, total_width, 3), (255, 255, 255), dtype=np.uint8)
        output_frame[:, :video_width] = frame_resized

        # Bodyparts
        for bp, cols in bodypart_columns.items():
            x_val, y_val = row[cols['x']], row[cols['y']]
            if pd.notna(x_val) and pd.notna(y_val):
                xd = int(float(x_val) * video_width / original_width)
                yd = int(float(y_val) * video_height / original_height)
                if 0 <= xd < video_width and 0 <= yd < video_height:
                    cv2.circle(output_frame, (xd, yd), 3, bodypart_colors.get(bp, (200, 200, 200)), -1)

        # Port labels on video
        for port_label, (px, py) in port_label_positions.items():
            label_color = port_colors.get(port_label, (90, 90, 90))
            lx = int(px * video_width / original_width)
            ly = int(py * video_height / original_height)
            text = str(port_label)
            font = cv2.FONT_HERSHEY_TRIPLEX
            (tw, th), baseline = cv2.getTextSize(text, font, 0.5, 1)
            x1, y1 = max(0, lx - 4), max(0, ly - th - 8)
            x2, y2 = min(video_width - 1, lx + tw + 6), min(video_height - 1, ly + baseline + 4)
            cv2.rectangle(output_frame, (x1, y1), (x2, y2), label_color, -1)
            cv2.putText(output_frame, text, (lx, ly), font, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
            timer_info = reward_icon_timers.get(port_label)
            if timer_info and timer_info.get('timer', 0) > 0:
                ix_pos = x2 + 20 if x2 + 20 <= video_width else x1 - 14
                iy_pos = (y1 + y2) // 2
                if timer_info.get('type') == 'drop':
                    px_size = int(timer_info.get('size', 26))
                    icon_to_use = icon_drop_resized.get(px_size, icon_drop_resized.get(26))
                else:
                    icon_to_use = icon_x_resized
                if icon_to_use is not None:
                    _overlay_icon(output_frame, (ix_pos, iy_pos), icon_to_use)

        current_node = _normalize_node(row[node_col]) if node_col is not None else None
        reward_size_value = row[reward_size_col] if reward_size_col is not None else np.nan

        if pd.notna(reward_size_value):
            target_port_label = None
            if current_node is not None:
                import re
                for p_node, p_label in port_map.items():
                    valid_nodes = {str(p_node).strip()}
                    m = re.search(r'\d+', str(p_label))
                    if m:
                        parent_data = reward_leaf_parents.get(int(m.group()))
                        if isinstance(parent_data, tuple) and len(parent_data) > 0:
                            valid_nodes.add(str(parent_data[0]).strip())
                            if len(parent_data) > 1 and isinstance(parent_data[1], list):
                                for gp in parent_data[1]:
                                    valid_nodes.add(str(gp).strip())
                    if str(current_node).strip() in valid_nodes:
                        target_port_label = p_label
                        break
            if target_port_label is None and reward_port_col is not None:
                raw_port = str(row[reward_port_col]).strip()
                if pd.notna(row[reward_port_col]) and raw_port.lower() not in {'nan', 'none', ''}:
                    if raw_port in port_map.values():
                        target_port_label = raw_port
                    elif raw_port in port_map.keys():
                        target_port_label = port_map[raw_port]
                    else:
                        import re
                        m = re.search(r'\d+', raw_port)
                        if m:
                            for lbl in port_map.values():
                                if m.group() in str(lbl):
                                    target_port_label = lbl
                                    break
            if target_port_label:
                val = float(reward_size_value)
                if val > 0:
                    reward_icon_timers[target_port_label] = {'timer': 40, 'type': 'drop', 'size': reward_size_to_px.get(val, 26)}
                else:
                    reward_icon_timers[target_port_label] = {'timer': 40, 'type': 'x', 'size': 24}

        # Graph edges
        for na, nb in G.edges():
            if na in graph_xy and nb in graph_xy:
                cv2.line(output_frame, graph_xy[na], graph_xy[nb], (55, 55, 55), 1)

        # Port nodes on graph
        for node_id, port_label in port_map.items():
            if node_id in graph_xy:
                xg, yg = graph_xy[node_id]
                nc = port_colors.get(port_label, (90, 90, 90))
                cv2.circle(output_frame, (xg, yg), 7, nc, 2)
                if port_label in {'Target1', 'Target2', 'Target3', 'Target4'}:
                    cv2.rectangle(output_frame, (xg - 20, yg - 30), (xg + 50, yg - 10), nc, -1)
                    cv2.putText(output_frame, str(port_label), (xg - 15, yg - 15), cv2.FONT_HERSHEY_TRIPLEX, 0.5, (255, 255, 255), 1)
                else:
                    cv2.rectangle(output_frame, (xg - 20, yg + 15), (xg + 50, yg + 35), nc, -1)
                    cv2.putText(output_frame, str(port_label), (xg - 15, yg + 30), cv2.FONT_HERSHEY_TRIPLEX, 0.5, (255, 255, 255), 1)

        # Reward icons on graph
        for node_id, port_label in port_map.items():
            info = reward_icon_timers.get(port_label)
            if info and info.get('timer', 0) > 0 and node_id in graph_xy:
                xg, yg = graph_xy[node_id]
                ixp = xg
                iyp = yg - 55 if port_label in {'Target1', 'Target2', 'Target3', 'Target4'} else yg + 65
                if info.get('type') == 'drop':
                    px_size = int(info.get('size', 26))
                    icon_to_use = icon_drop_resized.get(px_size, icon_drop_resized.get(26))
                else:
                    icon_to_use = icon_x_resized
                if icon_to_use is None:
                    continue
                ih, iw = icon_to_use.shape[:2]
                ixp = int(max(iw // 2, min(total_width - iw // 2, ixp)))
                iyp = int(max(ih // 2, min(total_height - ih // 2, iyp)))
                _overlay_icon(output_frame, (ixp, iyp), icon_to_use)

        # Current edge
        current_edge = _parse_edge_value(row[edge_col]) if edge_col is not None else None
        if current_edge is not None:
            ea, eb = current_edge
            if ea in graph_xy and eb in graph_xy:
                cv2.line(output_frame, graph_xy[ea], graph_xy[eb], current_color, 5)

        # Current node
        if current_node is not None and current_node in graph_xy:
            xg, yg = graph_xy[current_node]
            pl = port_map.get(current_node)
            bc = port_colors.get(pl, current_color) if pl else current_color
            cv2.circle(output_frame, (xg, yg), 9, current_color, -1)
            cv2.circle(output_frame, (xg, yg), 11, bc, 2)

        # === Change-of-mind annotations ===
        if change_of_mind_frame is not None:
            # Crosshair on apex node: orange before the event, red after
            if apex_node_id is not None and apex_node_id in graph_xy:
                ax_x, ax_y = graph_xy[apex_node_id]
                com_color = (0, 0, 255) if video_frame_idx >= change_of_mind_frame else (0, 140, 255)
                cv2.circle(output_frame, (ax_x, ax_y), 14, com_color, 2)
                cv2.line(output_frame, (ax_x - 17, ax_y), (ax_x + 17, ax_y), com_color, 2)
                cv2.line(output_frame, (ax_x, ax_y - 17), (ax_x, ax_y + 17), com_color, 2)

            # Persistent text banner on video from the event frame onward
            if video_frame_idx >= change_of_mind_frame:
                com_label = "CHANGE OF MIND"
                font = cv2.FONT_HERSHEY_DUPLEX
                (tw, th), _ = cv2.getTextSize(com_label, font, 0.7, 2)
                lx = (video_width - tw) // 2
                ly = 35
                cv2.rectangle(output_frame, (lx - 6, ly - th - 8), (lx + tw + 6, ly + 4), (0, 0, 0), -1)
                cv2.putText(output_frame, com_label, (lx, ly), font, 0.7, (0, 0, 255), 2, cv2.LINE_AA)

            # Red border flash fading over 2 s (80 frames) after the event
            frames_since_com = video_frame_idx - change_of_mind_frame
            if 0 <= frames_since_com <= 80:
                alpha_flash = max(0.0, 1.0 - frames_since_com / 80.0)
                overlay = output_frame.copy()
                cv2.rectangle(overlay, (0, 0), (video_width - 1, total_height - 1), (0, 0, 255), 8)
                cv2.addWeighted(overlay, alpha_flash, output_frame, 1.0 - alpha_flash, 0, output_frame)

        out.write(output_frame)

        for pl in list(reward_icon_timers.keys()):
            if reward_icon_timers[pl].get('timer', 0) > 0:
                reward_icon_timers[pl]['timer'] -= 1

    cap.release()
    out.release()
    print(f"✓ Saved to: {output_path}")

In [25]:
# Helper: convert MATLAB RGB colors to OpenCV BGR colors
# MATLAB typically stores colors as [R, G, B] with values in [0, 1]
# OpenCV expects (B, G, R) with values in [0, 255]

import numpy as np


def matlab_rgb_to_opencv(color):
    """Convert one MATLAB-style RGB color to an OpenCV BGR tuple.

    Parameters
    ----------
    color : sequence of 3 numbers
        MATLAB RGB values in [0, 1], ordered as [R, G, B].

    Returns
    -------
    tuple[int, int, int]
        OpenCV BGR values in 0-255 integer range.
    """
    rgb = np.asarray(color, dtype=float).reshape(3)
    rgb = np.clip(rgb, 0.0, 1.0)
    bgr = (rgb[::-1] * 255.0).round().astype(int)
    return tuple(int(v) for v in bgr)


def matlab_rgb_dict_to_opencv(color_dict):
    """Convert a dict of MATLAB RGB colors to OpenCV BGR colors."""
    return {key: matlab_rgb_to_opencv(value) for key, value in color_dict.items()}


# Example:
matlab_port_colors = {
    'Target1': [0.80, 0.25, 0.33],
    'Target2': [1.00, 0.55, 0.50],
    'Target3': [0.20, 0.45, 0.80],
    'Target4': [0.55, 0.75, 1.00],
    'Target5': [0.25, 0.60, 0.35],
    'Target6': [0.65, 0.80, 0.55],
    'Target7': [0.55, 0.40, 0.70],
    'Target8': [0.90, 0.60, 0.85]
}
port_colors = matlab_rgb_dict_to_opencv(matlab_port_colors)
print(port_colors)

{'Target1': (84, 64, 204), 'Target2': (128, 140, 255), 'Target3': (204, 115, 51), 'Target4': (255, 191, 140), 'Target5': (89, 153, 64), 'Target6': (140, 204, 166), 'Target7': (178, 102, 140), 'Target8': (217, 153, 230)}


In [ ]:
video_out_dir = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/indirect_path_videos")
video_out_dir.mkdir(parents=True, exist_ok=True)

for _, row in direct_opportunities_df.iterrows():
    start_nv = int(row['start_node_visit_idx'])
    end_nv = start_nv + int(row['path_seq_length_unique'])
    path_rows = improved_node_summary_df[
        (improved_node_summary_df['node_visit_idx'] >= start_nv) &
        (improved_node_summary_df['node_visit_idx'] <= end_nv)
    ]
    path_start_frame = int(path_rows.iloc[0]['start_frame'])
    path_end_frame = int(path_rows.iloc[-1]['end_frame'])

    import ast
    from_port, to_port = ast.literal_eval(row['path_pair_label'])
    change_of_mind_frame = int(row['direct_start_frame'])
    apex_node = row['direct_start_node']

    out_path = video_out_dir / f"path_{int(row['path_index']):02d}_{from_port}_to_{to_port}.mp4"

    print(f"\nPath {int(row['path_index'])}: {from_port} → {to_port} | apex={apex_node} | COM frame={change_of_mind_frame}")

    create_indirect_path_video(
        start_frame=path_start_frame,
        end_frame=path_end_frame,
        output_path=str(out_path),
        summary_df=summary_df,
        G=G,
        node_positions=node_positions,
        port_map=port_map,
        video_path=maze_video_path,
        change_of_mind_frame=change_of_mind_frame,
        apex_node_id=apex_node,
        reward_leaf_parents=reward_leaf_parents,
        fps=FPS,
        port_label_positions=port_label_positions,
        port_colors=port_colors
    )

print(f"\nAll {len(direct_opportunities_df)} indirect path videos saved to {video_out_dir}")


Path 0: Target5 → Target6 | apex=L519 | COM frame=2274
Found 5 bodyparts, reward_size_col=('reward_size', '')
Creating 866 frames...
✓ Saved to: /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/indirect_path_videos/path_00_Target5_to_Target6.mp4

Path 1: Target1 → Target8 | apex=R519 | COM frame=6042
Found 5 bodyparts, reward_size_col=('reward_size', '')
Creating 208 frames...
✓ Saved to: /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/indirect_path_videos/path_01_Target1_to_Target8.mp4

Path 2: Target2 → Target1 | apex=R518 | COM frame=26192
Found 5 bodyparts, reward_size_col=('reward_size', '')
Creating 427 frames...
✓ Saved to: /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/indirect_path_videos/path_02_Target2_to_Target1.mp4

Path 3: Target5 → Target4 | apex=L412 | COM frame=35856
Found 5 bodyparts, reward_size_col=('reward_size

KeyboardInterrupt: 